## Fe構造

**説明変数**

materials projectから取得した鉄の構造データを長周期構造に変換しています。このデータには目的変数は存在しません。

"../data/Fe4_xsf/" dirctoryに構造データ（xsf形式）が保存されています。

1. BCC

![](../data/Fe4_xsf/bcc.png)

2. FCC

![](../data/Fe4_xsf/fcc.png)

3. HCP

![](../data/Fe4_xsf/hcp.png)


以下に説明変数に変換したデータを保存しています。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.max_columns",60)
pd.set_option("display.max_rows",10)
df = pd.read_csv("../data_calculated/Fe2_descriptor.csv")
descriptor_names = [ 'a0.70_rp2.40', 'a0.70_rp3.00', 'a0.70_rp3.60', 'a0.70_rp4.20',
       'a0.70_rp4.80', 'a0.70_rp5.40',]
target_name = None
meta_names = ['key',  'polytype', 'id']

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist,squareform
from scipy.cluster.hierarchy import dendrogram, linkage
import copy
from scipy.stats import pearsonr
import numpy as np

def make_linkage(df, descriptor_names, target_name, corr="minus_abs_pearson"):
    labels = copy.deepcopy(descriptor_names)
    if target_name is not None:
        labels.append(target_name)
    Xraw = df.loc[:,labels].values
    scaler = StandardScaler()
    X = scaler.fit_transform(Xraw)
    df_tmp = pd.DataFrame(X)
    if corr=="minus_abs_pearson":
        corr = 1- np.abs(df_tmp.corr())
    else:
        raise ValueError("unknown corr={}".format(corr))
    pairdistance = squareform(corr)
    Z = linkage(pairdistance)
    return Z, labels

def show_dendrogram(Z,labels, corr):
    fig, ax = plt.subplots()
    dendrogram(Z,labels=labels,orientation="left", ax=ax)
    ax.set_xlabel(corr)
    fig.tight_layout()
    fig.show()
    
corr="minus_abs_pearson"
Z, labels = make_linkage(df, descriptor_names, target_name, corr)
show_dendrogram(Z,labels, corr)

In [ ]:
import os
import seaborn as sns
from copy import deepcopy

IMAGE_DIR = "image_keep"

imgfile = os.path.join(IMAGE_DIR, "Fe4_pairplot.png")
if not os.path.isfile(imgfile):
    alllabels = deepcopy(descriptor_names)
    img = sns.pairplot(df[alllabels])
    os.makedirs(IMAGE_DIR, exist_ok=True)
    img.savefig(imgfile)
    
from IPython import display
display.Image(imgfile)


**参考文献**

1. A. Jain$^*$, S.P. Ong$^*$, G. Hautier, W. Chen, W.D. Richards, S. Dacek, S. Cholia, D. Gunter, D. Skinner, G. Ceder, K.A. Persson (*=equal contributions), 
"The Materials Project: A materials genome approach to accelerating materials innovation", 
APL Materials, 2013, 1(1), 011002.
doi:10.1063/1.4812323